# 34 — Model Serving & Freshness

### Why this exists

A model can be correct while the serving system is wrong if features are stale, transformed differently or missing.

### Learning objectives

By the end you should be able to:
- explain the mechanism in plain language;
- predict a system behavior before running the experiment;
- measure one observable consequence;
- diagnose one controlled failure;
- state the production implication.

### Prerequisite

Use the preceding notebook in this sequence as your conceptual starting point. You do not need a vendor account for this notebook.

### Engineering loop

**Why → Sketch → Predict → Build → Measure → Break → Diagnose → Improve → Production**


## Capability contract

**Capability:** C07 — ML Reliability  
**Workstream:** ML Platform  
**Primary roles:** ML Engineer|Data Engineer  
**You will prove:** Separate offline signal from production behavior and define validation


## 🖊️ Sketch note

![Sketch note — Model Serving & Freshness](../assets/sketch-notes/34-model-serving-freshness.svg)

> **Visual-first rule:** explain the system in your own words before reading the code.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S11 — 1138: DATA-1138 — Churn score uses stale customer activity** → [`../work_simulations/11_acme_commerce_feature_freshness/README.md`](../work_simulations/11_acme_commerce_feature_freshness/README.md)
- **S13 — 1152: DATA-1152 — Online churn scores disagree with batch scores** → [`../work_simulations/13_acme_commerce_serving_skew/README.md`](../work_simulations/13_acme_commerce_serving_skew/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Predict before you run

**Question:** Will offline and online features represent the same customer state?

Write your prediction and the reason in your own words before running the code.


In [ ]:
import pandas as pd
features=pd.DataFrame({"customer_id":[1,2,3],"orders_30d_offline":[5,2,8],"orders_30d_online":[5,1,8]})
features["skew"]=features.orders_30d_online-features.orders_30d_offline
print(features)
assert features.loc[features.customer_id==2,"skew"].iloc[0]==-1


## Measure

Record the output that tests your prediction. If you can, change exactly one input and compare before/after behavior.


## Break deliberately

Introduce one controlled failure: malformed input, stale data, wrong split, missing context, unauthorized tool, or another failure relevant to this notebook. Keep the failure small and observable.


In [ ]:
# Break: change one online feature without updating the offline feature definition.
features_bad=features.copy()
features_bad.loc[1,"orders_30d_online"] += 3
features_bad["skew"]=features_bad.orders_30d_online-features_bad.orders_30d_offline
print(features_bad)
assert features_bad.skew.abs().max()>0


## Diagnose

Write:
1. **Symptom** — what changed?
2. **Evidence** — what observation supports it?
3. **Mechanism** — why did it happen?
4. **Fix** — what is the smallest defensible improvement?


## Improve

Make the smallest change that prevents or exposes the failure. Re-run the same measurement so the comparison is fair.


## Production bridge

A production system also needs ownership, versioning, observability, security/authorization, cost or latency budgets, failure recovery and a rollback path. Name the two that matter most here and why.


## Explain it in 60–90 seconds

Explain the mechanism, one surprising failure mode, and one production trade-off without relying on vendor names.


## Independent challenge

Build a second experiment around **batch vs online features**. Change one assumption, measure the result, and defend the design choice.

### Evidence to keep

- prediction
- measurement
- failure evidence
- diagnosis
- improvement
- production implication


## Measure freshness

In [ ]:
from datetime import datetime,timedelta,timezone
now=datetime(2026,9,30,tzinfo=timezone.utc)
feature_time=now-timedelta(minutes=7)
age=(now-feature_time).total_seconds()/60
print("feature age minutes:",age)
assert age==7

## Break the serving contract

In [ ]:
max_age=5
fresh=age<=max_age
print("fresh enough:",fresh)
assert not fresh